## fetch.py

### Calling the API

In [1]:
from dotenv import load_dotenv
import requests, json, time, os

load_dotenv()

api_key = os.getenv("football_data_api_key")


### Saved local raw data for matches

In [2]:
matches_url = "https://api.football-data.org/v4/competitions/PL/matches"
headers = {"X-Auth-Token": api_key}

os.makedirs("data/raw", exist_ok=True)

for season in [2023, 2024, 2025]:
    path = f"data/raw/matches_{season}.json"
    if os.path.exists(path):          
        continue
    r = requests.get(matches_url, headers=headers, params={"season": season})
    print(season, r.status_code)
    r.raise_for_status()
    with open(path, "w") as f:
        json.dump(r.json(), f)
    time.sleep(7) 

### Current matches need to be fetched each time

In [3]:
matches_url = "https://api.football-data.org/v4/competitions/PL/matches"
headers = {"X-Auth-Token": api_key}

os.makedirs("data/raw", exist_ok=True)

path = "data/raw/matches_2026.json"
r = requests.get(matches_url, headers=headers)
print(path, r.status_code)
r.raise_for_status()
with open(path, "w") as f:
    json.dump(r.json(), f)

data/raw/matches_2026.json 200


### Current standings need to be fetched each time

In [4]:
standings_url = "https://api.football-data.org/v4/competitions/PL/standings"
headers = {"X-Auth-Token": api_key}

os.makedirs("data/raw", exist_ok=True)

path = "data/raw/standings.json"
r = requests.get(standings_url, headers=headers)
print(path, r.status_code)
r.raise_for_status()
with open(path, "w") as f:
    json.dump(r.json(), f, indent=2)

data/raw/standings.json 200


## clean.py

### Clean matches SQL

In [ ]:
import duckdb

con = duckdb.connect("data/grader.duckdb")

con.sql("""
    CREATE OR REPLACE TABLE matches_clean AS
    SELECT
        m.season.id AS season_id,
        m.matchday AS match_day,
        m.id AS match_id,
        m.utcDate AS date,
        m.homeTeam.name AS home_team,
        m.awayTeam.name AS away_team,
        m.score.winner AS winner,
        m.status AS status
    FROM (
        SELECT unnest(matches) AS m
        FROM read_json_auto('data/raw/matches_*.json')
    )

""")

con.sql("SELECT * FROM matches_clean").show()

┌───────────┬───────────┬──────────┬─────────────────────┬───────────────────────────┬────────────────────────────┬───────────┬──────────┐
│ season_id │ match_day │ match_id │        date         │         home_team         │         away_team          │  winner   │  status  │
│   int64   │   int64   │  int64   │      timestamp      │          varchar          │          varchar           │  varchar  │ varchar  │
├───────────┼───────────┼──────────┼─────────────────────┼───────────────────────────┼────────────────────────────┼───────────┼──────────┤
│      1564 │         1 │   435943 │ 2023-08-11 19:00:00 │ Burnley FC                │ Manchester City FC         │ AWAY_TEAM │ FINISHED │
│      1564 │         1 │   435944 │ 2023-08-12 12:00:00 │ Arsenal FC                │ Nottingham Forest FC       │ HOME_TEAM │ FINISHED │
│      1564 │         1 │   435945 │ 2023-08-12 14:00:00 │ AFC Bournemouth           │ West Ham United FC         │ DRAW      │ FINISHED │
│      1564 │         1 │  

### Clean standings SQL

In [ ]:
import duckdb

con.sql("""
    CREATE OR REPLACE TABLE standings_clean AS
    SELECT
        t.team.name AS team,
        t.position AS position,
        t.points AS points,
        t.goalDifference AS goal_difference,
        t.playedGames as games_played
    FROM (
        SELECT unnest(standings[1].table) AS t
        FROM read_json_auto('data/raw/standings.json')
    )
""")

con.sql("SELECT * FROM standings_clean").show()

┌───────────────────────────┬──────────┬────────┬─────────────────┬──────────────┐
│           team            │ position │ points │ goal_difference │ games_played │
│          varchar          │  int64   │ int64  │     int128      │    int64     │
├───────────────────────────┼──────────┼────────┼─────────────────┼──────────────┤
│ Manchester City FC        │        1 │     15 │               8 │            5 │
│ Arsenal FC                │        2 │     12 │               4 │            5 │
│ Brighton & Hove Albion FC │        3 │     10 │              11 │            5 │
│ Brentford FC              │        4 │      9 │               6 │            5 │
│ Leeds United FC           │        5 │      9 │               4 │            5 │
│ Liverpool FC              │        6 │      9 │               3 │            5 │
│ Everton FC                │        7 │      9 │               3 │            5 │
│ Hull City AFC             │        8 │      8 │               2 │            5 │
│ Ne

## Metrics.py

### Form SQL

In [ ]:
con.sql("""
    CREATE OR REPLACE TABLE form_last_five_games AS
    WITH x as (
    Select
        match_id,
        date,
        home_team as team,
        away_team as opponent,
        case when winner = 'HOME_TEAM' then 'win'
            when winner = 'DRAW' then 'draw'
            else 'loss'
        end as result,
        case when winner = 'HOME_TEAM' then 3
            when winner = 'DRAW' then 1
            else 0
        end as points
    From matches_clean
    Where date <= CURRENT_DATE

    Union all

    Select
        match_id,
        date,
        away_team as team,
        home_team as opponent,
        case when winner = 'AWAY_TEAM' then 'win'
            when winner = 'DRAW' then 'draw'
            else 'loss'
        end as result,
        case when winner = 'AWAY_TEAM' then 3
            when winner = 'DRAW' then 1
            else 0
        end as points
    From matches_clean
    Where date <= CURRENT_DATE
    ),

    ranked as (
        select *,
            row_number() over (partition by team order by date desc) as rn
        from x
    )

    select team, sum(points) as form
    from ranked
    where rn <= 5
    group by team
""")

con.sql("SELECT * FROM form_last_five_games").show()

┌────────────────────────────┬────────┐
│            team            │  form  │
│          varchar           │ int128 │
├────────────────────────────┼────────┤
│ Hull City AFC              │      8 │
│ Liverpool FC               │      9 │
│ Southampton FC             │      1 │
│ Everton FC                 │      9 │
│ Tottenham Hotspur FC       │      2 │
│ Aston Villa FC             │      4 │
│ Luton Town FC              │      1 │
│ Sheffield United FC        │      0 │
│ AFC Bournemouth            │      3 │
│ Manchester United FC       │      5 │
│          ·                 │      · │
│          ·                 │      · │
│          ·                 │      · │
│ Wolverhampton Wanderers FC │      3 │
│ Crystal Palace FC          │      4 │
│ Coventry City FC           │      3 │
│ Burnley FC                 │      2 │
│ Brighton & Hove Albion FC  │     10 │
│ West Ham United FC         │      6 │
│ Brentford FC               │      9 │
│ Chelsea FC                 │      7 │


### H2H SQL

In [ ]:
con.sql("""
    CREATE OR REPLACE TABLE h2h_form AS
    WITH y as (
    Select
        match_id,
        date,
        home_team as team,
        away_team as opponent,
        case when winner = 'HOME_TEAM' then 'win'
            when winner = 'DRAW' then 'draw'
            else 'loss'
        end as result,
        case when winner = 'HOME_TEAM' then 3
            when winner = 'DRAW' then 1
            else 0
        end as points
    From matches_clean
    Where date <= CURRENT_DATE

    Union all

    Select
        match_id,
        date,
        away_team as team,
        home_team as opponent,
        case when winner = 'AWAY_TEAM' then 'win'
            when winner = 'DRAW' then 'draw'
            else 'loss'
        end as result,
        case when winner = 'AWAY_TEAM' then 3
            when winner = 'DRAW' then 1
            else 0
        end as points
    From matches_clean
    Where date <= CURRENT_DATE
    ),

    ranked as (
        select *,
            row_number() over (partition by team, opponent order by date desc) as rn
        from y
    )

    select team, opponent, avg(points) as h2h_form_pl, count(*) as meetings
    from ranked
    where rn <= 6
    group by team, opponent
""")

con.sql("SELECT * FROM h2h_form").show()

┌──────────────────────┬────────────────────────────┬────────────────────┬──────────┐
│         team         │          opponent          │    h2h_form_pl     │ meetings │
│       varchar        │          varchar           │       double       │  int64   │
├──────────────────────┼────────────────────────────┼────────────────────┼──────────┤
│ AFC Bournemouth      │ West Ham United FC         │                1.0 │        6 │
│ Tottenham Hotspur FC │ Manchester United FC       │ 1.8333333333333333 │        6 │
│ Nottingham Forest FC │ Brentford FC               │ 1.6666666666666667 │        6 │
│ Crystal Palace FC    │ Tottenham Hotspur FC       │                1.5 │        6 │
│ AFC Bournemouth      │ Burnley FC                 │                2.0 │        4 │
│ Fulham FC            │ Wolverhampton Wanderers FC │ 1.6666666666666667 │        6 │
│ Sheffield United FC  │ Liverpool FC               │                0.0 │        2 │
│ Burnley FC           │ Everton FC                 │ 

### Fixture grader final table

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE fixture_grader AS
WITH fixtures AS (
    SELECT match_id, date, home_team AS team, away_team AS opponent, TRUE AS is_home
    FROM matches_clean
    WHERE status IN ('SCHEDULED', 'TIMED')

    UNION ALL

    SELECT match_id, date, away_team AS team, home_team AS opponent, FALSE AS is_home
    FROM matches_clean
    WHERE status IN ('SCHEDULED', 'TIMED')
),

next_fixture AS (
    SELECT *
    FROM (
        SELECT *,
            row_number() OVER (PARTITION BY team ORDER BY date, match_id) AS rn
        FROM fixtures
    )
    WHERE rn = 1
),

standings_scored AS (
    SELECT
        team,
        points,
        goal_difference,
        points / (3.0 * games_played) AS league_score,
        (count(*) OVER () - rank() OVER (ORDER BY goal_difference DESC))
        / (count(*) OVER () - 1.0) AS gd_score
    FROM standings_clean
),

scaled AS (
    SELECT
        n.team,
        n.opponent,
        n.is_home,
        s.points AS opp_points,
        s.goal_difference AS opp_goal_difference,
        f.form AS opp_form,
        h.h2h_form_pl AS h2h_ppg,
        f.form / 15.0 AS form_score,
        s.league_score,
        s.gd_score,
        1 - COALESCE(h.h2h_form_pl / 3.0, 0.5) AS h2h_score,
        CASE WHEN n.is_home THEN 0.36 ELSE 0.64 END AS venue_score
    FROM next_fixture n
    LEFT JOIN standings_scored s ON n.opponent = s.team
    LEFT JOIN form_last_five_games f ON n.opponent = f.team
    LEFT JOIN h2h_form h ON n.team = h.team AND n.opponent = h.opponent
)

SELECT *,
    ROUND(100 * (0.30 * form_score
               + 0.30 * league_score
               + 0.10 * gd_score
               + 0.20 * h2h_score
               + 0.10 * venue_score), 1) AS difficulty
FROM scaled
""")

con.sql("SELECT * FROM fixture_grader ORDER BY difficulty DESC").show()

┌───────────────────────────┬───────────────────────────┬─────────┬────────────┬─────────────────────┬──────────┬────────────────────┬─────────────────────┬─────────────────────┬─────────────────────┬─────────────────────┬──────────────┬────────────┐
│           team            │         opponent          │ is_home │ opp_points │ opp_goal_difference │ opp_form │      h2h_ppg       │     form_score      │    league_score     │      gd_score       │      h2h_score      │ venue_score  │ difficulty │
│          varchar          │          varchar          │ boolean │   int64    │       int128        │  int128  │       double       │       double        │       double        │       double        │       double        │ decimal(3,2) │   double   │
├───────────────────────────┼───────────────────────────┼─────────┼────────────┼─────────────────────┼──────────┼────────────────────┼─────────────────────┼─────────────────────┼─────────────────────┼─────────────────────┼──────────────┼──────────